Train/test split

Prices rise every year (eda.ipynb section 2.1), so a random split would let the model see future price levels. The split is **time-based**: bookings are sorted by arrival date, the oldest ~80% are used for training and the newest ~20% (cut at a whole month) are kept as the test set. The test set is not used until the final evaluation.

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 67
TARGET = "ADR"
TEST_SIZE = 0.2

df = pd.read_csv("H2_regression.csv")
print(df.shape)

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

# helper year-month only used for sorting, it is not a feature
# (ArrivalDateDayOfMonth was dropped so bookings are ordered by month)
month_num = df["ArrivalDateMonth"].map({m: i + 1 for i, m in enumerate(MONTHS)})
period = pd.PeriodIndex.from_fields(year=df["ArrivalDateYear"], month=month_num, freq="M")
df = df.assign(_period=period).sort_values("_period", kind="stable").reset_index(drop=True)

# cut at a month boundary so no month is shared between train and test
cut_month = df["_period"].iloc[int(len(df) * (1 - TEST_SIZE))]
is_test = df["_period"] >= cut_month
train, test = df[~is_test], df[is_test]

print(f"train: {len(train):,} rows ({len(train) / len(df):.0%})  {train['_period'].min()} to {train['_period'].max()}")
print(f"test : {len(test):,} rows ({len(test) / len(df):.0%})  {test['_period'].min()} to {test['_period'].max()}")
print("median ADR train/test:", train[TARGET].median(), test[TARGET].median())

train, test = train.drop(columns="_period"), test.drop(columns="_period")

Both files stay sorted by arrival date so time-ordered cross-validation (TimeSeriesSplit) can be used on the training set.

In [ ]:
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)
print("saved train.csv", train.shape, " test.csv", test.shape)